In [ ]:
from pathlib import Path
import csv

from PIL import Image
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq

def clean_text(text: str) -> str:
    block_to_remove = """system
You are a helpful assistant.
user
You are a clinical image assistant.

Look at the image and output ONLY the anatomical location.
- If an ADE is present: output only the location of the ADE.
- If no ADE: output only the body part shown.
- Output must be a short phrase (1–3 words), no explanation strictly.

Examples:
Image: (oral ulcers)
Response: The anatomical location of the ADE is in the tongue

Image: (swollen leg from drug edema)
Response: The anatomical location of the ADE is in the left leg

Image: (abdominal rash from ADE)
Response: The anatomical location of the ADE is in the stomach

Image: (no ADE, only arm visible)
Response: The anatomical location of the ADE is in the right arm

Image: (swollen leg from drug edema)
Response: The anatomical location of the ADE is in the right leg

Image: (no ADE, face visible)
Response: The anatomical location of the ADE is in the face 
assistant
The anatomical location of the ADE is in the """

    # Remove only if the text starts with this block
    if text.startswith(block_to_remove):
        return text[len(block_to_remove):].lstrip()
    return text

def run_qwen25_vl_on_folder(
    model_id: str,
    image_dir: str,
    output_csv: str,
    prompt: str,
    max_new_tokens: int = 128,
):
    """
    Run a Qwen2.5-VL model on all images in a folder and save outputs to CSV.

    CSV format:
        imageName, generatedText, Prompt
    - Header row as above.
    - Second row has the prompt text only once in the "Prompt" column.
    - Subsequent rows: one per image with imageName & generatedText,
      and empty Prompt column.
    """

    print(f"Loading model: {model_id}")

    # Processor + vision-text model
    processor = AutoProcessor.from_pretrained(
        model_id,
        trust_remote_code=True
    )

    model = AutoModelForVision2Seq.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()

    image_path = Path(image_dir)
    if not image_path.exists():
        raise FileNotFoundError(f"Input folder not found: {image_path.resolve()}")

    # Collect image files
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}
    image_files = sorted(
        [p for p in image_path.iterdir() if p.suffix.lower() in exts]
    )

    if not image_files:
        raise ValueError(f"No image files found in {image_path.resolve()}")

    print(f"Found {len(image_files)} images.")

    # Open CSV for writing
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        # Header
        writer.writerow(["imageName", "generatedText", "Prompt"])

        # Row with prompt only once
        writer.writerow(["", "", prompt])

        # Process each image
        for idx, img_path in enumerate(image_files, start=1):
            print(f"[{idx}/{len(image_files)}] Processing {img_path.name} ...")

            try:
                image = Image.open(img_path).convert("RGB")

                # Qwen2.5-VL chat template: image + text
                messages = [
                    {
                        "role": "user",
                        "content": [
                            {"type": "image"},
                            {"type": "text", "text": prompt},
                        ],
                    }
                ]

                text_input = processor.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                )

                inputs = processor(
                    text=[text_input],
                    images=[image],
                    return_tensors="pt",
                ).to(model.device)

                with torch.no_grad():
                    output_ids = model.generate(
                        **inputs,
                        max_new_tokens=max_new_tokens,
                    )

                generated = processor.batch_decode(
                    output_ids,
                    skip_special_tokens=True
                )[0].strip()

                #print(f"RAW Response: {generated}")

                generated = clean_text(generated)

                print(f"Response: {generated}")
                # Write row: prompt column left empty after first time
                writer.writerow([img_path.name, generated, ""])

            except Exception as e:
                err_msg = f"ERROR: {type(e).__name__}: {e}"
                print(f"  -> Failed on {img_path.name}: {err_msg}")
                writer.writerow([img_path.name, err_msg, ""])

    print(f"Done. CSV written to: {output_csv}")


In [ ]:
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
image_dir = "./data/LLaVA-Med/images/multimodal adr"   # your folder
output_csv = "./data/LLaVA-Med/Qwen2_Localization_D1500_Output_V1.csv"
prompt = ( 
    """You are a clinical image assistant.

Look at the image and output ONLY the anatomical location.
- If an ADE is present: output only the location of the ADE.
- If no ADE: output only the body part shown.
- Output must be a short phrase (1–3 words), no explanation strictly.

Examples:
Image: (oral ulcers)
Response: The anatomical location of the ADE is in the tongue

Image: (swollen leg from drug edema)
Response: The anatomical location of the ADE is in the left leg

Image: (abdominal rash from ADE)
Response: The anatomical location of the ADE is in the stomach

Image: (no ADE, only arm visible)
Response: The anatomical location of the ADE is in the right arm

Image: (swollen leg from drug edema)
Response: The anatomical location of the ADE is in the right leg

Image: (no ADE, face visible)
Response: The anatomical location of the ADE is in the face """ )

run_qwen25_vl_on_folder(
    model_id=model_id,
    image_dir=image_dir,
    output_csv=output_csv,
    prompt=prompt,
    max_new_tokens=256,
)
